# AI Assisted Side Channel Analysis Lecture

## Disclaimer!!!

Lots of this lecture builds off other open-source material, not limited to
- [ChipWhisperer Jupyter Notebook Repository](https://github.com/newaetech/chipwhisperer-jupyter)
- [SimpleCrypto SCALIB](https://github.com/simple-crypto/SCALib)
- [Google SCAAML and Sedpack](https://github.com/google/scaaml)
- [Hardware hacker Handbook Tutorials](https://github.com/HardwareHackingHandbook/notebooks/tree/main/labs)

Please follow all licenses for each material, and only distribute / use code from this lesson as an educational tool.
Please understand and follow the LICENSE file in this repo.

### Software Needed for Lesson
- python >= 3.11 (I'm using 3.12)
    - Highly recommend you use a python virtual environment tool (Mamba, Conda, Pip venv)
- Jupyter tool for notebook (I'm using labs, but a pure notebook should work)
- Python libraries (all dependencies will install from these). Just use `pip install <package-name>`
    - cwtraces
    - chipwhisperer
    - scalib
- Highly recommend giving ChipWhisperer a peak
    - Great intro tutorials for many hardware attacks https://github.com/newaetech/chipwhisperer-jupyter


### Papers to Read
Many great conferences, some good journals.
The International Association for Cryptologic Research (IACR) is a leader in the field, and has multiple proceedings.
Lots of focus on open-source results, and many great software/hardware demos
- [Generalized Power Attacks against Crypto Hardware using Long-Range Deep Learning](https://tches.iacr.org/index.php/TCHES/article/view/11685)
- [TPUXtract: An Exhaustive Hyperparameter Extraction Framework](https://tches.iacr.org/index.php/TCHES/article/view/11923)
- [A Comparison of Multi-task Learning and Single-Task Learning Approaches](https://iacr.steepath.eu/2023/611-AComparisonofMultitasklearningandSingletasklearningApproaches.pdf)
- [Non-Profiled Deep Learning-based Side-Channel attacks with Sensitivity Analysis](https://tches.iacr.org/index.php/TCHES/article/view/7387)

### Great Books
- [Embedded Cryptography 1, 2 and 3](https://www.wiley.com/en-us/Embedded+Cryptography+1-p-9781394351862)
- [The Hardware Hacking Handbook](https://nostarch.com/hardwarehacking)

### Other Datasets and Benchmarks for SCA
- ASCAD (v1 and v2)
- CHES Challenege 2024
- ChipWhisperer Open-source Datasets
- TinyAES
- Simulated datasets (this tutorial)

# Simple Power Analysis and Hamming Weights

In [ ]:
#imports
import numpy as np
import random
import pylab as py
import matplotlib.pyplot as plt

# Make this lab repeatable
random.seed(42) 
np.random.seed(42)

#Generate a random lookup table as sbox
lookup = [x for x in range(256)]
random.shuffle(lookup)

# Create a simulated power trace, based on data in (din)
def measure_power(din):
    #secret byte
    skey = 0b101010 # 0x42
    
    #Calculate result
    res = lookup[din ^ skey]
    
    #Generate some arbitrary random data, not necessarily around zero
    b = 148
    a = 154    
    basetrace = (b - a) * np.random.random_sample(50) + a
    
    #Determine number of 1's in result (Hamming Weight), create appropriately sized spike
    time_of_leakage = 35
    basetrace[time_of_leakage] += bin(res).count('1') - 4 # On average, HW=4, so we minus it out
        
    return basetrace

In [ ]:
# Create a number of simulated traces and input data
def gen_traces(number_traces):    
    traces = np.array([None] * number_traces)
    input_data = np.random.randint(0, 256, number_traces)

    for d in range(0, number_traces):
        traces[d] = np.array(measure_power(input_data[d]))
    return (input_data, traces)
    
(input_data, traces) = gen_traces(1000)

In [ ]:
#what is the data?

print(f"{input_data.shape=}")
print(f"{traces.shape=}")
print(f"{traces[0].shape=}\n")

#some sample input data
print(input_data[:10])
print(input_data[500:510],'\n')

#some sample trace data
print(traces[10],'\n')
print(np.mean(traces, axis=0))

#some sample plots

In [ ]:
#Lets plot a traces
plt.figure()

#create a base x array, of all the time steps.
temp_x = np.arange(50)

plt.plot(temp_x, traces[0])

In [ ]:
#Lets plot a few of the traces
plt.figure()

subset = np.random.choice(temp_x, size=5)

for i in subset:
    plt.plot(temp_x, traces[i])

In [ ]:
#Lets plot all the traces
plt.figure()

for i in range(10):
    plt.plot(temp_x, traces[i])

In [ ]:
#plot the average of all the traces
trace_avg = np.mean(traces)

plt.figure()
plt.plot(temp_x, trace_avg)

## Better Data
Let's load in some more emulated data, that actually matches an AES implementation.
This data is taken from the ChipWhisperer Jupyter notebook tutorials, and are fully simulated traces
First, let's re-run all of our plots, but after we load some better data.

In [ ]:
#load in some better data
from cwtraces import sca101_lab_data

data = sca101_lab_data["lab3_1"]()
trace_array  =  data["trace_array"]
textin_array = data["textin_array"]

trace_array_x = np.arange(trace_array.shape[1])

print(f'{trace_array.shape=}')
print(f'{textin_array.shape=}')

In [ ]:
# Plot a single trace
plt.plot(trace_array_x, trace_array[0])

In [ ]:
#Lets plot a few of the traces
plt.figure()

subset = np.random.choice(temp_x, size=5)

for i in subset:
    plt.plot(trace_array_x, trace_array[i])

In [ ]:
#Lets plot all the traces
plt.figure()

for i in range(10):
    plt.plot(trace_array_x, trace_array[i])

In [ ]:
#Finally, lets look at the average
plt.plot(trace_array_x, np.mean(trace_array, axis=0))

In [ ]:
#How close is a random trace to the average?
plt.figure()
plt.plot(trace_array_x, trace_array[50], 'b')

In [ ]:
#Let's look directly at the difference between a single trace, and the mean
plt.figure()
plt.plot(trace_array_x, np.mean(trace_array, axis=0) - trace_array[50])

# HW
Lets do some math, and build off what we know.
Is there a clear difference in traces, based on these values?

In [ ]:
#Let's look at all the HWs of 0-255
HW = [bin(n).count("1") for n in range(0, 256)]
print(HW)

For a simple test, lets just split the dataset in two, and remove all traces that use an input with the first byte as 0.
And then let's grab all the others, i.e. $HW=(1,8)$, and store them together.
Is there a difference between the two sets?

In [ ]:
#split arrays into 0s and 1s
ones = []
zeros = []

byte = 0

for i in range(trace_array.shape[0]):
    if textin_array[i][byte] == 0:
        zeros.append(trace_array[i])
    else:
        ones.append(trace_array[i])

zeros = np.array(zeros)
ones = np.array(ones)

zeros_avg = np.mean(zeros, axis=0)
ones_avg = np.mean(ones, axis=0)

In [ ]:
plt.figure()
plt.plot(trace_array_x, zeros_avg)
plt.figure()
plt.plot(trace_array_x, ones_avg)
plt.figure()
plt.plot(trace_array_x, zeros_avg, 'k', ones_avg)

In [ ]:
#so, some differences, but clearly not too many. maybe we subtract them from each other?
plt.plot(trace_array_x, zeros_avg - ones_avg)
plt.legend(["0s - 1s"])

These spikes at the start and near the middle help reflect where data is being manipulated in this trace.
The peaks show the variations. 
Data related to byte 0 and IV 0 is near the front.
Operations relating to byte 0 IV 1 is close to the middle

# SNR

In [ ]:
from scalib.metrics import SNR

snr = SNR(nc=256)

print(trace_array.shape, textin_array.shape)

#need to multiple by 100, to translate into int16, to fit library
snr.fit_u((100*trace_array).astype(np.int16), textin_array.astype(np.uint16))
snr_val = snr.get_snr()
print(snr_val.shape)

plt.figure()
plt.plot(snr_val)
plt.legend(np.arange(16))
plt.show()

# TVLA
Test Vector Leakage Assessment (TVLA) can assess a target's vulnerability to power analysis via a generic test.

Just to save time, you can look more into TVLA via the following notebook, from ChipWhisperer.
They layout why TVLA is used, the standard way to perform TVLA, and show examples.
[link](https://github.com/newaetech/chipwhisperer-jupyter/blob/main/courses/sca203/Introduction%20to%20TVLA.ipynb)
I will pull in some of their test here, to describe the theory.

The basic idea is to collect two sets of power traces that we expect to have different means based only on side channel leakage.
We can then assess the likelyhood that their means are actually different, or only different due to variance in the power traces.
To assess this likelyhood, we'll use Welsh's T-Test:

$$
t = \frac{\bar{X_1} - \bar{X_2}}{\sqrt{\frac{s_1^2}{N_1} + \frac{s_2^2}{N_2}}}
$$

What data should we use for our two sets of power traces?
A simple set is fixed vs. random text.
The first set of data is a constant key and a constant plaintext, giving an on average constant non-zero leakage.
The other is a fixed key (the same as the first group) with a random plaintext, giving an average leakage near zero.
Rambus has a [document](https://www.rambus.com/wp-content/uploads/2015/08/TVLA-DTR-with-AES.pdf) outlining how to perform TVLA tests.
In it, they specify what values to use for the fixed plaintext and key:

$$
    I_{fixed} = \texttt{0xda39a3ee5e6b4b0d3255bfef95601890} \\
    K_{dev} = \texttt{0x0123456789abcdef123456789abcdef0}
$$

as well as the random plaintext/fixed key:

$$
    K_{gen} = \texttt{0x123456789abcdef123456789abcde0f0} \\
    I_{0} = \texttt{0x00000000000000000000000000000000} \\
    I_{j+1} = \mathtt{AES(I_{j}, K_{gen})} \\
    K_{dev} = \texttt{0x0123456789abcdef123456789abcdef0}
$$

# CPA

Use Pearson's Coefficient to find the key, by measuring the correlation of all key guesses 

## Taken from ChipWhisperer SCA 101 Lab 4.2
[link](https://github.com/newaetech/chipwhisperer-jupyter/blob/main/courses/sca101/SOLN_Lab%204_2%20-%20CPA%20on%20Firmware%20Implementation%20of%20AES.ipynb)

Developing our Correlation Algorithm


We'll be testing how good our guess is using a measurement called the Pearson correlation coefficient, which measures the linear correlation between two datasets.
The actual algorithm is as follows for datasets $X$ and $Y$ of length $N$, with means of $\bar{X}$ and $\bar{Y}$, respectively: 
    $$r = \frac{cov(X, Y)}{\sigma_X \sigma_Y}$$


Dataset here can mean a lot, just think of them as two sets of independent data right now.
Eventually, these two datasets will be the hamming weights of our guesses and our traces.

$cov(X, Y)$ is the covariance of `X` and `Y` and can be calculated as follows:
$$cov(X, Y) = \sum_{n=1}^{N}[(Y_n - \bar{Y})(X_n - \bar{X})]$$

$\sigma_X$ and $\sigma_Y$ are the standard deviation of the two datasets. This value can be calculated with the following equation:

$$\sigma_X = \sqrt{\sum_{n=1}^{N}(X_n - \bar{X})^2}$$

As you can see, the calculation is actually broken down pretty nicely into some smaller chunks that we can implement with some simple functions. While we could use a library to calculate all this stuff for us, being able to implement a mathematical algorithm in code is a useful skill to develop. 

To start, build the following functions:

1. `mean(X)` to calculate the mean of a dataset
1. `std_dev(X, X_bar)` to calculate the standard deviation of a dataset. We'll need to reuse the mean for the covariance, so it makes more sense to calculate it once and pass it in to each function
1. `cov(X, X_bar, Y, Y_bar)` to calculate the covariance of two datasets. Again, we can just pass in the means we calculate for std_dev here.

**HINT: You can use `np.sum(X, axis=0)` to replace all of the $\sum$ from earlier. The argument `axis=0` will sum across columns, allowing us to use a single `mean`, `std_dev`, and `cov` call for the entire power trace**"


In [ ]:
def mean(X):
    return np.sum(X, axis=0)/len(X)

def std_dev(X, X_bar):
    return np.sqrt(np.sum((X-X_bar)**2, axis=0))

def cov(X, X_bar, Y, Y_bar):
    return np.sum((X-X_bar)*(Y-Y_bar), axis=0)

In [ ]:
from tqdm.notebook import trange
from src.aes_helper import aes_internal, HW

#creates a blank list of 0s
maxcpa = [0] * 256

# we don't need to redo the mean and std dev calculations 
# for each key guess
t_bar = mean(trace_array) 
o_t = std_dev(trace_array, t_bar)

#loop through all key guesses
for kguess in range(0, 256):
    #grab the HWs of all inputs, for the sole key guess
    hws = np.array([[HW[aes_internal(textin[0], kguess)] for textin in textin_array]]).transpose()

    #mean and std of all the HW for this key guess
    hws_bar = mean(hws)
    o_hws = std_dev(hws, hws_bar)

    #correlate the hws of the key guess to the traces
    correlation = cov(trace_array, t_bar, hws, hws_bar)
    #this is the pearson's coefficent
    cpa_output = correlation/(o_t*o_hws)
    #save the highest person's coefficient for this guess.
    maxcpa[kguess] = max(abs(cpa_output))
    
#Let me introduce you to your best friend, Argmax!
guess = np.argmax(maxcpa)
guess_corr = max(maxcpa)

print("Key guess: ", hex(guess), guess)
print("Correlation: ", guess_corr)

# Template - LDA
For time-sakes, lets just take a peak at this good example of an LDA walkthrough, for SCA.
[Link](https://github.com/simple-crypto/SCALib/blob/main/examples/aes_attack.py)

# Template - Neural Networks
We are going to heavily pull from Google on this, and their work over using ML for SCA template attacks. [Link](https://github.com/google/scaaml/blob/main/scaaml_intro/key_recovery_demo.ipynb)

NOTE: This repo will not work in the same python repo, it needs an older version of python, using Tensorflow 2.6.
We will just walk through it together.